# *Time- and preference-aware recommender transformer*

In [17]:
import re
import sys
import os
import scrapbook as sb
from tempfile import TemporaryDirectory
import numpy as np
import pandas as pd

from collections import defaultdict
import tensorflow as tf
tf.get_logger().setLevel('ERROR') # only show error messages

#from recommenders.utils.timer import Timer
#from recommenders.datasets.amazon_reviews import get_review_data
#from recommenders.datasets.split_utils import filter_k_core

# Transformer Based Models
#from recommenders.models.sasrec.model import SASREC
#from recommenders.models.sasrec.ssept import SSEPT

# Sampler for sequential prediction
#from recommenders.models.sasrec.sampler import WarpSampler
#from recommenders.models.sasrec.util import SASRecDataSet

print("System version: {}".format(sys.version))
print("Tensorflow version: {}".format(tf.__version__))

System version: 3.11.14 (main, Oct 21 2025, 18:31:21) [GCC 11.2.0]
Tensorflow version: 2.15.0


In [37]:
from timeit import default_timer


class Timer(object):
    """Timer class.

    `Original code <https://github.com/miguelgfierro/pybase/blob/2298172a13fb4a243754acbc6029a4a2dcf72c20/log_base/timer.py>`_.

    Examples:
        >>> import time
        >>> t = Timer()
        >>> t.start()
        >>> time.sleep(1)
        >>> t.stop()
        >>> t.interval < 1
        True
        >>> with Timer() as t:
        ...   time.sleep(1)
        >>> t.interval < 1
        True
        >>> "Time elapsed {}".format(t) #doctest: +ELLIPSIS
        'Time elapsed 1...'
    """

    def __init__(self):
        self._timer = default_timer
        self._interval = 0
        self.running = False

    def __enter__(self):
        self.start()
        return self

    def __exit__(self, *args):
        self.stop()

    def __str__(self):
        return "{:0.4f}".format(self.interval)

    def start(self):
        """Start the timer."""
        self.init = self._timer()
        self.running = True

    def stop(self):
        """Stop the timer. Calculate the interval in seconds."""
        self.end = self._timer()
        try:
            self._interval = self.end - self.init
            self.running = False
        except AttributeError:
            raise ValueError(
                "Timer has not been initialized: use start() or the contextual form with Timer() as t:"
            )

    @property
    def interval(self):
        """Get time interval in seconds.

        Returns:
            float: Seconds.
        """
        if self.running:
            raise ValueError("Timer has not been stopped, please use stop().")
        else:
            return self._interval

####sasrec class(Custom)

In [49]:
# Copyright (c) Microsoft Corporation. All rights reserved.
# Licensed under the MIT License.
import random
import numpy as np
from tqdm import tqdm
import tensorflow as tf

#from recommenders.utils.timer import Timer


class MultiHeadAttention(tf.keras.layers.Layer): #tf.keras.layers.Layer 클래스 상속
    """
    - Q (query), K (key) and V (value) are split into multiple heads (num_heads)
    - each tuple (q, k, v) are fed to scaled_dot_product_attention
    - all attention outputs are concatenated
    """

    def __init__(self, attention_dim, num_heads, dropout_rate): #MultiHeadAttention(attention_dim, num_heads, dropout_rate)로 불러옴

        """MultiHeadAttention클래스의 args 초기화

        Args:
            attention_dim (int): 어텐션 임베딩의 차원
            num_heads (int): 멀티헤드어텐션 모듈의 개수
            dropout_rate (float): 드롭아웃 비율
        """
        super(MultiHeadAttention, self).__init__() #상속받은 부모클래스(tf.keras.layers.Layer)의 메서드나 속성을 사용하게 해줌
        self.num_heads = num_heads
        self.attention_dim = attention_dim
        assert attention_dim % self.num_heads == 0 #attention_dim이 self.num_heads로 정확하게 나누어 떨어지는지 확인, 나누어 떨어지지않으면 예외 발생
        self.dropout_rate = dropout_rate

        self.depth = attention_dim // self.num_heads

        self.Q = tf.keras.layers.Dense(self.attention_dim, activation=None) #attention_dim만큼의 노드를 가지고있는 dense layer 정의
        self.K = tf.keras.layers.Dense(self.attention_dim, activation=None)
        self.V = tf.keras.layers.Dense(self.attention_dim, activation=None)
        
        self.S_Dense = tf.keras.layers.Dense(1, activation=None, name="S_Scaler") # 평점 임베딩을 스칼라 가중치 S로 변환

        self.dropout = tf.keras.layers.Dropout(self.dropout_rate)


    def call(self, queries, keys, rating): #클래스의 인스턴스를 생성후 그 인스턴스를 호출할때 넣는 인자 정의
        """Model forward pass.

        Args:
            queries (tf.Tensor): 쿼리의 텐서
            keys (tf.Tensor): 키의 텐서
            rating (tf.Tensor): 키(keys)에 해당하는 평점 임베딩 텐서

        Returns:
            tf.Tensor: Output tensor.
        """

        # Linear projections
        Q = self.Q(queries)  # (N, T_q, C)
        K = self.K(keys)  # (N, T_k, C)
        V = self.V(keys)  # (N, T_k, C)
        

        S_scalar = self.S_Dense(rating) 

        # --- MULTI HEAD ---
        # Split and concat, Q_, K_ and V_ are all (h*N, T_q, C/h)
        Q_ = tf.concat(tf.split(Q, self.num_heads, axis=2), axis=0) 
        K_ = tf.concat(tf.split(K, self.num_heads, axis=2), axis=0)
        V_ = tf.concat(tf.split(V, self.num_heads, axis=2), axis=0)
        

        


        # S_scalar (N, T_k, 1) -> S_tiled (h*N, T_k, 1)
        S_tiled = tf.tile(S_scalar, [self.num_heads, 1, 1])
        # S_tiled (h*N, T_k, 1) -> S_broadcastable (h*N, 1, T_k)
        S_broadcastable = tf.transpose(S_tiled, [0, 2, 1])
        
        # --- SCALED DOT PRODUCT ---
        # Multiplication (QK^T)
        outputs = tf.matmul(Q_, tf.transpose(K_, [0, 2, 1]))  # (h*N, T_q, T_k) #Q * Kt

        # (h*N, T_q, T_k) * (h*N, 1, T_k) -> (h*N, T_q, T_k)
        outputs = outputs * S_broadcastable

        # Scale
        outputs = outputs / (K_.get_shape().as_list()[-1] ** 0.5) #k차원의 제곱근으로 나눠줌

        # Key Masking
        key_masks = tf.sign(tf.abs(tf.reduce_sum(keys, axis=-1)))  # (N, T_k)
        key_masks = tf.tile(key_masks, [self.num_heads, 1])  # (h*N, T_k)
        key_masks = tf.tile(
            tf.expand_dims(key_masks, 1), [1, tf.shape(queries)[1], 1]
        )  # (h*N, T_q, T_k)

        paddings = tf.ones_like(outputs) * (-(2 ** 32) + 1) 
        outputs = tf.where(tf.equal(key_masks, 0), paddings, outputs) 

        # Future blinding (Causality)
        diag_vals = tf.ones_like(outputs[0, :, :])  # (T_q, T_k)
        tril = tf.linalg.LinearOperatorLowerTriangular( #diag_vals의 하삼각 행렬
            diag_vals
        ).to_dense()  # (T_q, T_k)
        masks = tf.tile(
            tf.expand_dims(tril, 0), [tf.shape(outputs)[0], 1, 1] 
        )  # (h*N, T_q, T_k)

        paddings = tf.ones_like(masks) * (-(2 ** 32) + 1) 
        outputs = tf.where(tf.equal(masks, 0), paddings, outputs)

        # Activation
        outputs = tf.nn.softmax(outputs)  # (h*N, T_q, T_k) #output에 softmax 함수 적용

        # Query Masking
        query_masks = tf.sign(tf.abs(tf.reduce_sum(queries, axis=-1))) 
        query_masks = tf.tile(query_masks, [self.num_heads, 1])  
        query_masks = tf.tile(
            tf.expand_dims(query_masks, -1), [1, 1, tf.shape(keys)[1]] 
        )  # (h*N, T_q, T_k)
        outputs *= query_masks  

        # Dropouts
        outputs = self.dropout(outputs) #드롭아웃 적용

        # Weighted sum
        outputs = tf.matmul(outputs, V_)  # ( h*N, T_q, C/h) #
        
        # --- MULTI HEAD ---
        # concat heads
        outputs = tf.concat(
            tf.split(outputs, self.num_heads, axis=0), axis=2
        )  #(h*N, T_q, C/h)-> 스플릿으로 (N,T_q, C/h) -> concat으로 2번쨰 차원에 이어붙여서 -> (N, T_q, C)

        # Residual connection
        outputs += queries

        return outputs


class MultiHeadAttention2(tf.keras.layers.Layer): #tf.keras.layers.Layer 클래스 상속
    """
    - Q (query), K (key) and V (value) are split into multiple heads (num_heads)
    - each tuple (q, k, v) are fed to scaled_dot_product_attention
    - all attention outputs are concatenated
    """

    def __init__(self, attention_dim, num_heads, dropout_rate): #MultiHeadAttention(attention_dim, num_heads, dropout_rate)로 불러옴

        """MultiHeadAttention클래스의 args 초기화

        Args:
            attention_dim (int): 어텐션 임베딩의 차원
            num_heads (int): 멀티헤드어텐션 모듈의 개수
            dropout_rate (float): 드롭아웃 비율
        """
        super(MultiHeadAttention2, self).__init__() #상속받은 부모클래스(tf.keras.layers.Layer)의 메서드나 속성을 사용하게 해줌
        self.num_heads = num_heads
        self.attention_dim = attention_dim
        assert attention_dim % self.num_heads == 0 #attention_dim이 self.num_heads로 정확하게 나누어 떨어지는지 확인, 나누어 떨어지지않으면 예외 발생
        self.dropout_rate = dropout_rate

        self.depth = attention_dim // self.num_heads

        self.Q = tf.keras.layers.Dense(self.attention_dim, activation=None) #attention_dim만큼의 노드를 가지고있는 dense layer 정의
        self.K = tf.keras.layers.Dense(self.attention_dim, activation=None)
        self.V = tf.keras.layers.Dense(self.attention_dim, activation=None)
        #self.R_V = tf.keras.layers.Dense(self.attention_dim, activation=None)
        self.dropout = tf.keras.layers.Dropout(self.dropout_rate)


    def call(self, queries, keys): #클래스의 인스턴스를 생성후 그 인스턴스를 호출할때 넣는 인자 정의 / attention_layer = MultiHeadAttention(attention_dim=512, num_heads=8, dropout_rate=0.1) -> output = attention_layer(queries, keys)
                                   #인스턴스에서 호출할때는 queries, keys를 넣어야하며 얘내들은 텐서여야함
                                   #queries랑 keys의 마지막차원을 인스턴스 정의할때 설정한 attention dim과 맞춰야함
        """Model forward pass.

        Args:
            queries (tf.Tensor): 쿼리의 텐서
            keys (tf.Tensor): 키의 텐서

        Returns:
            tf.Tensor: Output tensor.
        """

        # Linear projections
        Q = self.Q(queries)  # (N, T_q, C) #queries는 self.Q를 통과한후 각각의 "샘플"에대해 attention_dim개의 출력값이 있는 텐서로 변환됨 / 만약 queries의 형태가 (batch_size, input_dim)이라면, self.Q(queries)의 결과인 Q의 형태는 (batch_size, attention_dim)
        K = self.K(keys)  # (N, T_k, C)
        V = self.V(keys)  # (N, T_k, C)
        #R_V = self.R_V(rating)
        # --- MULTI HEAD ---
        # Split and concat, Q_, K_ and V_ are all (h*N, T_q, C/h)
        Q_ = tf.concat(tf.split(Q, self.num_heads, axis=2), axis=0) #Q행렬을 마지막 차원을 기준으로 self.num_heads의 수만큼 분할 / (batch_size, sequence_length, attention_dim/num_heads) -> 분할된 행렬들을 첫 번째 차원을 기준으로 연결 / (batch_size*num_heads, sequence_length, attention_dim/num_heads)
        K_ = tf.concat(tf.split(K, self.num_heads, axis=2), axis=0)
        V_ = tf.concat(tf.split(V, self.num_heads, axis=2), axis=0)
        #R_V_ = tf.concat(tf.split(R_V, self.num_heads, axis=2), axis=0)

        # --- SCALED DOT PRODUCT ---
        # Multiplication
        outputs = tf.matmul(Q_, tf.transpose(K_, [0, 2, 1]))  # (h*N, T_q, T_k) #Q * Kt

        # Scale
        outputs = outputs / (K_.get_shape().as_list()[-1] ** 0.5) #k차원의 제곱근으로 나눠줌

        # Key Masking
        # 입력시퀀스에 0으로 패딩된 것들에 가중치 안주려고
        key_masks = tf.sign(tf.abs(tf.reduce_sum(keys, axis=-1)))  # (N, T_k) #패딩이 아닌 원소는1, 패딩인 원소는 0
        key_masks = tf.tile(key_masks, [self.num_heads, 1])  # (h*N, T_k) #num_heads수 만큼 마스크 복제(세로)
        key_masks = tf.tile(
            tf.expand_dims(key_masks, 1), [1, tf.shape(queries)[1], 1] #queries의 길이만큼 마스크 복제(가로)
        )  # (h*N, T_q, T_k)

        paddings = tf.ones_like(outputs) * (-(2 ** 32) + 1) #패딩된 위치에 attention score를 매누 낮게 설정함으로써 패딩 위치에 대한 attention을 주지 않도록 함
        # outputs, (h*N, T_q, T_k)
        outputs = tf.where(tf.equal(key_masks, 0), paddings, outputs) #key_masks의 각 텐서의 원소가 0이면 padding의 해당 위치값(attention score 낮게 주는거), 아니면 outputs 해당 위치 값 사용

        # 여기까지 사각행렬(outputs)이 정의되고

        # Future blinding (Causality)
        diag_vals = tf.ones_like(outputs[0, :, :])  # (T_q, T_k) / output의 첫번쨰 배치와 같은 shape의 1로 이루어진 텐서(diag_vals) 생성
        tril = tf.linalg.LinearOperatorLowerTriangular( #diag_vals의 하삼각 행렬(주대각선 위 0 아래 그대로) 생성후 밀집행렬로 변환
            diag_vals
        ).to_dense()  # (T_q, T_k)
        masks = tf.tile(
            tf.expand_dims(tril, 0), [tf.shape(outputs)[0], 1, 1] #tril의 0번째 차원에 새 차원 추가 , output의 0번째 shape -> tril텐서를 3차원으로 확장후 output의 0번째 shape만큼 늘리기 -> tril shape: (output 0번쨰 차원, tril의 1번쨰 차원, tril의 2번째 차원) -> masks
        )  # (h*N, T_q, T_k)                                      #위에서 만들어진 1로 이루어진 하삼각행렬(tril)을 output의 배치수만큼 복사

        paddings = tf.ones_like(masks) * (-(2 ** 32) + 1) # (output 0번쨰 차원, tril의 1번쨰 차원, tril의 2번째 차원) shape의 1로 이루어진 텐서 * 큰 음수
        # outputs, (h*N, T_q, T_k)
        outputs = tf.where(tf.equal(masks, 0), paddings, outputs) #masks가 0인곳에는 패딩값, 아닌곳에서는 outputs -> outputs이 하삼각행렬(미래정보를 볼수없는)로 변환됨

        #여기서 위에 정의된 사각행렬을 하삼각행렬로 변환

        #outputs *= R_V_
        # Activation
        outputs = tf.nn.softmax(outputs)  # (h*N, T_q, T_k) #output에 softmax 함수 적용

        # Query Masking, query_masks (N, T_q)
        """
        reduce_sum(queries, axis = -1)
        queries = [
          [
            [1, 2, 3],
            [4, 5, 6]
          ],
          [
            [7, 8, 9],
            [10, 11, 12]
          ]
        ]

        result = [
          [6, 15],
          [24, 33]
        ]

        """
        query_masks = tf.sign(tf.abs(tf.reduce_sum(queries, axis=-1))) #sign: 양수면1 0이면 0 음수면 -1
        query_masks = tf.tile(query_masks, [self.num_heads, 1])  # query_mask의 shpae(N(배치길이), T_q(시퀀스 길이)) -> h(num_heads)만큼 복제 -> (h*N, T_q)
        query_masks = tf.tile(
            tf.expand_dims(query_masks, -1), [1, 1, tf.shape(keys)[1]] #query_masks의 마지막차원에 1 추가 -> (h*N, T_q, 1) -> [1,1, tf.shape(keys)[1]] (1부분(0번쨰와 1번쨰 차원)은 그대로 두고, 2번째 차원을 keys의 1번째 shape만큼 복사)
        )  # (h*N, T_q, T_k)
        outputs *= query_masks  # broadcasting. (N, T_q, C) #output은 어텐션점수 포함, 패딩된 위치에대한 어텐션값을 0으로 설정

        # Dropouts
        outputs = self.dropout(outputs) #드롭아웃 적용

        # Weighted sum
        outputs = tf.matmul(outputs, V_with_rating) # V_ -> V_with_rating
        #outputs = tf.matmul(outputs, R_V_)
        #outputs = outputs * R_V_


        # --- MULTI HEAD ---
        # concat heads
        outputs = tf.concat(
            tf.split(outputs, self.num_heads, axis=0), axis=2
        )  #(h*N, T_q, C/h)-> 스플릿으로 (N,T_q, C/h) -> concat으로 2번쨰 차원에 이어붙여서 -> (N, T_q, C)

        # Residual connection
        outputs += queries

        return outputs


class PointWiseFeedForward(tf.keras.layers.Layer):
    """
    Convolution layers with residual connection
    """

    def __init__(self, conv_dims, dropout_rate):
        """Initialize parameters.

        Args:
            conv_dims (list): List of the dimensions of the Feedforward layer.
            dropout_rate (float): Dropout probability.
        """
        super(PointWiseFeedForward, self).__init__()
        self.conv_dims = conv_dims
        self.dropout_rate = dropout_rate
        self.conv_layer1 = tf.keras.layers.Conv1D(
            filters=self.conv_dims[0], kernel_size=1, activation="relu", use_bias=True #출력의 차원수를 conv_dims[0]으로
        )
        self.conv_layer2 = tf.keras.layers.Conv1D(
            filters=self.conv_dims[1], kernel_size=1, activation=None, use_bias=True
        )
        self.dropout_layer = tf.keras.layers.Dropout(self.dropout_rate)

    def call(self, x):
        """Model forward pass.

        Args:
            x (tf.Tensor): Input tensor.

        Returns:
            tf.Tensor: Output tensor.
        """
        #x : (배치사이즈, 시퀀스렝스, 피쳐개수)
        output = self.conv_layer1(x) #(배치사이즈, 시퀀스렝스, conv_dims[0])(relu함수 적용)
        output = self.dropout_layer(output) #드롭아웃 적용

        output = self.conv_layer2(output) #(배치사이즈, 시퀀스렝스, conv_dims[1])(그대로 나옴)
        output = self.dropout_layer(output) #드롭아웃 적용

        # Residual connection
        output += x

        return output


class EncoderLayer(tf.keras.layers.Layer):
    """
    Transformer based encoder layer
    셀프어텐션, ffn 다 합친거
    """

    def __init__(
        self,
        seq_max_len,
        embedding_dim,



        
        attention_dim,
        num_heads,
        conv_dims,
        dropout_rate,
    ):
        """Initialize parameters.

        Args:
            seq_max_len (int): Maximum sequence length.
            embedding_dim (int): Embedding dimension.
            attention_dim (int): Dimension of the attention embeddings.
            num_heads (int): Number of heads in the multi-head self-attention module.
            conv_dims (list): List of the dimensions of the Feedforward layer.
            dropout_rate (float): Dropout probability.
        """
        super(EncoderLayer, self).__init__()

        self.seq_max_len = seq_max_len
        self.embedding_dim = embedding_dim #인코더레이어(인코더)단계에선 임베딩이없어, 인코더 전 단계에서 임베딩이 진행됨
                                           #여기에 임베딩 차원을 따로 정의하는 이유는 밑에 사용자정의 정규화함수에 사용되기때문

        self.mha = MultiHeadAttention(attention_dim, num_heads, dropout_rate)
        self.ffn = PointWiseFeedForward(conv_dims, dropout_rate)

        self.layernorm1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)

        self.dropout1 = tf.keras.layers.Dropout(dropout_rate)
        self.dropout2 = tf.keras.layers.Dropout(dropout_rate)

        self.layer_normalization = LayerNormalization(
            self.seq_max_len, self.embedding_dim, 1e-08
        )

    def call_(self, x, training, mask):
        """Model forward pass.

        Args:
            x (tf.Tensor): Input tensor.
            training (tf.Tensor): Training tensor.
            mask (tf.Tensor): Mask tensor.

        Returns:
            tf.Tensor: Output tensor.
        """

        attn_output = self.mha(queries=self.layer_normalization(x), keys=x)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(x + attn_output)

        # feed forward network
        ffn_output = self.ffn(out1)  # (batch_size, input_seq_len, d_model)
        ffn_output = self.dropout2(ffn_output, training=training)
        out2 = self.layernorm2(
            out1 + ffn_output
        )  # (batch_size, input_seq_len, d_model)

        # masking
        out2 *= mask

        return out2

    def call(self, x, rating, training, mask):
        """Model forward pass.

        Args:
            x (tf.Tensor): Input tensor.
            training (tf.Tensor): Training tensor.
            mask (tf.Tensor): Mask tensor.

        Returns:
            tf.Tensor: Output tensor.
        """

        x_norm = self.layer_normalization(x) #input을 정규화(사용자 정의 함수)
        rating_norm = self.layer_normalization(rating)
        attn_output = self.mha(queries=x_norm, keys=x, rating = rating_norm) #mha의 쿼리는 input정규화한거, 키는 input / 원래 트랜스포머는 쿼리 키 밸류 둘다 같은값에서 나오는데 sasrec은 다르게 처리한듯?
                                                       #쿼리가 x_norm, 키랑 밸류가 x에서 나옴
        attn_output = self.ffn(attn_output) #output을 피드포워드 통과
        out = attn_output * mask #output을 mask랑 곱(행렬곱x)

        return out


class Encoder(tf.keras.layers.Layer):
    """
    Invokes Transformer based encoder with user defined number of layers
    위에서 정의한 인코더 레이어를 num_layers만큼 쌓는거
    """

    def __init__(
        self,
        num_layers,
        seq_max_len,
        embedding_dim,
        attention_dim,
        num_heads,
        conv_dims,
        dropout_rate,
    ):
        """Initialize parameters.

        Args:
            num_layers (int): Number of layers.
            seq_max_len (int): Maximum sequence length.
            embedding_dim (int): Embedding dimension.
            attention_dim (int): Dimension of the attention embeddings.
            num_heads (int): Number of heads in the multi-head self-attention module.
            conv_dims (list): List of the dimensions of the Feedforward layer.
            dropout_rate (float): Dropout probability.
            위의 인코더레이어의 args와 num_layers 유무차이(레이어를 몇개 쌓을지 정하는 class이기떄문에)
        """
        super(Encoder, self).__init__()

        self.num_layers = num_layers

        self.enc_layers = [
            EncoderLayer(
                seq_max_len,
                embedding_dim,
                attention_dim,
                num_heads,
                conv_dims,
                dropout_rate,
            )
            for _ in range(num_layers)
        ]

        self.dropout = tf.keras.layers.Dropout(dropout_rate)

    def call(self, x, rating, training, mask):
        """Model forward pass.

        Args:
            x (tf.Tensor): Input tensor. -> 시퀀스임베딩(포지셔널임베딩까지 더한거)
            training (tf.Tensor): Training tensor.
            mask (tf.Tensor): Mask tensor.

        Returns:
            tf.Tensor: Output tensor.
        """

        for i in range(self.num_layers):
            x = self.enc_layers[i](x,rating, training, mask)

        return x  # (batch_size, input_seq_len, d_model)





class DecoderLayer(tf.keras.layers.Layer):
    """
    Transformer based encoder layer
    셀프어텐션, ffn 다 합친거
    """

    def __init__(
        self,
        seq_max_len,
        embedding_dim,
        attention_dim,
        num_heads,
        conv_dims,
        dropout_rate,
    ):
        """Initialize parameters.

        Args:
            seq_max_len (int): Maximum sequence length.
            embedding_dim (int): Embedding dimension.
            attention_dim (int): Dimension of the attention embeddings.
            num_heads (int): Number of heads in the multi-head self-attention module.
            conv_dims (list): List of the dimensions of the Feedforward layer.
            dropout_rate (float): Dropout probability.
        """
        super(DecoderLayer, self).__init__()

        self.seq_max_len = seq_max_len
        self.embedding_dim = embedding_dim #인코더레이어(인코더)단계에선 임베딩이없어, 인코더 전 단계에서 임베딩이 진행됨
                                           #여기에 임베딩 차원을 따로 정의하는 이유는 밑에 사용자정의 정규화함수에 사용되기때문

        self.mha = MultiHeadAttention(attention_dim, num_heads, dropout_rate)
        self.mha2 = MultiHeadAttention2(attention_dim, num_heads, dropout_rate)
        self.ffn = PointWiseFeedForward(conv_dims, dropout_rate)

        self.layernorm1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)

        self.dropout1 = tf.keras.layers.Dropout(dropout_rate)
        self.dropout2 = tf.keras.layers.Dropout(dropout_rate)

        self.layer_normalization = LayerNormalization(
            self.seq_max_len, self.embedding_dim, 1e-08
        )
#self.decoder(seq_attention, seq_embeddings2, seq_embeddings_rating2, tratining, mask,look_ahead_mask)
    def call(self, seq_attention, seq_embeddings2, seq_embeddings_rating2, training, mask,look_ahead_mask):
        """Model forward pass.

        Args:
            x (tf.Tensor): Input tensor.
            training (tf.Tensor): Training tensor.
            mask (tf.Tensor): Mask tensor.

        Returns:
            tf.Tensor: Output tensor.
        """

        x_norm = self.layer_normalization(seq_embeddings2) #input을 정규화(사용자 정의 함수)
        rating_norm = self.layer_normalization(seq_embeddings_rating2)

        encoder_output_norm = self.layer_normalization(seq_attention) #input을 정규화(사용자 정의 함수)

        #첫번쨰서브층(룩어헤드) -> 쿼리키밸류: 인풋
        #attn_output = self.mha(queries=x_norm, keys=seq_embeddings2, rating = rating_norm)
        #attn_output = self.dropout1(attn_output, training=training)
        #out1 = self.layernorm1(x_norm + attn_output)
        #out1 *= look_ahead_mask

        #두번째 서브층(패딩마스크) -> 쿼리: out1, 키밸류: 인코더아웃풋
        attn_output2 = self.mha2(queries=encoder_output_norm, keys=x_norm)
        attn_output2 = self.dropout1(attn_output2, training=training)
        out2 = self.layernorm1(attn_output2 + encoder_output_norm)
        out2 *= look_ahead_mask

        # feed forward network
        ffn_output = self.ffn(out2)  # (batch_size, input_seq_len, d_model)
        ffn_output = self.dropout2(ffn_output, training=training)
        out3 = self.layernorm2(
            out2 + ffn_output
        )  # (batch_size, input_seq_len, d_model)

        return out3

    def call_(self, x, rating, training, mask):
        """Model forward pass.

        Args:
            x (tf.Tensor): Input tensor.
            training (tf.Tensor): Training tensor.
            mask (tf.Tensor): Mask tensor.

        Returns:
            tf.Tensor: Output tensor.
        """

        x_norm = self.layer_normalization(x) #input을 정규화(사용자 정의 함수)
        rating_norm = self.layer_normalization(rating)
        attn_output = self.mha(queries=x_norm, keys=x, rating = rating_norm) #mha의 쿼리는 input정규화한거, 키는 input / 원래 트랜스포머는 쿼리 키 밸류 둘다 같은값에서 나오는데 sasrec은 다르게 처리한듯?
                                                       #쿼리가 x_norm, 키랑 밸류가 x에서 나옴
        attn_output = self.ffn(attn_output) #output을 피드포워드 통과
        out = attn_output * mask #output을 mask랑 곱(행렬곱x)

        return out


class Decoder(tf.keras.layers.Layer):
    """
    Invokes Transformer based encoder with user defined number of layers
    위에서 정의한 인코더 레이어를 num_layers만큼 쌓는거
    """

    def __init__(
        self,
        num_layers,
        seq_max_len,
        embedding_dim,
        attention_dim,
        num_heads,
        conv_dims,
        dropout_rate,
    ):
        """Initialize parameters.

        Args:
            num_layers (int): Number of layers.
            seq_max_len (int): Maximum sequence length.
            embedding_dim (int): Embedding dimension.
            attention_dim (int): Dimension of the attention embeddings.
            num_heads (int): Number of heads in the multi-head self-attention module.
            conv_dims (list): List of the dimensions of the Feedforward layer.
            dropout_rate (float): Dropout probability.
            위의 인코더레이어의 args와 num_layers 유무차이(레이어를 몇개 쌓을지 정하는 class이기떄문에)
        """
        super(Decoder, self).__init__()

        self.num_layers = num_layers

        self.dec_layers = [
            DecoderLayer(
                seq_max_len,
                embedding_dim,
                attention_dim,
                num_heads,
                conv_dims,
                dropout_rate,
            )
            for _ in range(num_layers)
        ]

        self.dropout = tf.keras.layers.Dropout(dropout_rate)

    def call(self, seq_attention, seq_embeddings2, seq_embeddings_rating2, training, mask,look_ahead_mask):
        """Model forward pass.

        Args:
            x (tf.Tensor): Input tensor. -> 시퀀스임베딩(포지셔널임베딩까지 더한거)
            training (tf.Tensor): Training tensor.
            mask (tf.Tensor): Mask tensor.

        Returns:
            tf.Tensor: Output tensor.
        """

        for i in range(self.num_layers):
            x = self.dec_layers[i](seq_attention, seq_embeddings2, seq_embeddings_rating2, training, mask,look_ahead_mask)

        return x  # (batch_size, input_seq_len, d_model)






class LayerNormalization(tf.keras.layers.Layer):
    """
    Layer normalization using mean and variance
    gamma and beta are the learnable parameters
    """

    def __init__(self, seq_max_len, embedding_dim, epsilon):
        """Initialize parameters.

        Args:
            seq_max_len (int): Maximum sequence length.
            embedding_dim (int): Embedding dimension.
            epsilon (float): Epsilon value.
        """
        super(LayerNormalization, self).__init__()
        self.seq_max_len = seq_max_len
        self.embedding_dim = embedding_dim
        self.epsilon = epsilon
        self.params_shape = (self.seq_max_len, self.embedding_dim)
        g_init = tf.ones_initializer()
        self.gamma = tf.Variable(
            initial_value=g_init(shape=self.params_shape, dtype="float32"),
            trainable=True,
        )
        b_init = tf.zeros_initializer()
        self.beta = tf.Variable(
            initial_value=b_init(shape=self.params_shape, dtype="float32"),
            trainable=True,
        )


    def call(self, x):
        """Model forward pass.

        Args:
            x (tf.Tensor): Input tensor. (Shape: B, S, D)
                           S(시퀀스 길이)는 self.seq_max_len(50)과 다를 수 있음 (e.g., 5)

        Returns:
            tf.Tensor: Output tensor.
        """
        # x shape: (B, S, D) (e.g., 1, 5, 128)
        # self.gamma shape: (50, 128)
        
        # 1. 입력 텐서의 동적 시퀀스 길이(S)를 가져옴
        current_seq_len = tf.shape(x)[1] # e.g., 5

        # 2. 정규화 수행 (normalized shape: B, S, D)
        mean, variance = tf.nn.moments(x, [-1], keepdims=True)
        normalized = (x - mean) / ((variance + self.epsilon) ** 0.5)
        
        # 3. self.gamma와 self.beta를 동적 시퀀스 길이에 맞게 슬라이싱
        
        # e.g., self.gamma[50-5:, :] -> self.gamma[45:, :] -> shape (5, 128)
        gamma_sliced = self.gamma[self.seq_max_len - current_seq_len :, :]
        beta_sliced = self.beta[self.seq_max_len - current_seq_len :, :]

        # 4. 브로드캐스팅을 통해 곱셈/덧셈 수행
        #    (B, S, D) * (S, D) -> (B, S, D)
        output = gamma_sliced * normalized + beta_sliced
        return output


class SASREC(tf.keras.Model):
    """SAS Rec model
    Self-Attentive Sequential Recommendation Using Transformer

    :Citation:

        Wang-Cheng Kang, Julian McAuley (2018), Self-Attentive Sequential
        Recommendation. Proceedings of IEEE International Conference on
        Data Mining (ICDM'18)

        Original source code from nnkkmto/SASRec-tf2,
        https://github.com/nnkkmto/SASRec-tf2

    """

    def __init__(self, **kwargs): #kwargs: 지정된값이 있을때 값이랑 없을때 값을 따로 지정할떄 사용가능 -> 디폴트값을 정해준다
        """Model initialization.

        Args:
            item_num (int): Number of items in the dataset.
            seq_max_len (int): Maximum number of items in user history.
            num_blocks (int): Number of Transformer blocks to be used.
            embedding_dim (int): Item embedding dimension.
            attention_dim (int): Transformer attention dimension.
            conv_dims (list): List of the dimensions of the Feedforward layer.
            dropout_rate (float): Dropout rate.
            l2_reg (float): Coefficient of the L2 regularization.
            num_neg_test (int): Number of negative examples used in testing.
        """
        super(SASREC, self).__init__()

        self.item_num = kwargs.get("item_num", None) #item_num이 지정된값이 있으면 그거쓰고 없으면 None사용
        self.seq_max_len = kwargs.get("seq_max_len", 100)
        self.num_blocks = kwargs.get("num_blocks", 2)
        self.embedding_dim = kwargs.get("embedding_dim", 100) #여기서 임베딩 차원수를 지정해줄때 item2vec으로 몇차원으로 임베딩했는지를 맞춰줘야할듯, 그래야 사용자정의 정규화함수가 제대로 작동됨
        self.attention_dim = kwargs.get("attention_dim", 100)
        self.attention_num_heads = kwargs.get("attention_num_heads", 1)
        self.conv_dims = kwargs.get("conv_dims", [100, 100])
        self.dropout_rate = kwargs.get("dropout_rate", 0.5)
        self.l2_reg = kwargs.get("l2_reg", 0.0)
        self.num_neg_test = kwargs.get("num_neg_test", 100)
        self.recent_item = kwargs.get("recent_imte", 5)

        self.max_interval = kwargs.get("max_interval", 2001) # fallback으로 2001 사용
        '''
        self.item_embedding_layer_i2v = tf.keras.layers.Embedding(
            self.item_num + 1,
            self.embedding_dim,
            embeddings_initializer=tf.keras.initializers.Constant(item2vec_embedding_matrix),
            name="item_embeddings",
            mask_zero=True,
            embeddings_regularizer=tf.keras.regularizers.L2(self.l2_reg),
        )
        '''
        self.item_embedding_layer = tf.keras.layers.Embedding(
            self.item_num + 1,                                  #item_num+1: 아이템 임베딩레이어 입력차원,
            self.embedding_dim,                                 #embedding_dim: 아이템 임베딩레이어 출력차원
            name="item_embeddings",                             #name = "item_embeddings": 모델의 다른부분과 구분하기위해 아이템 임베딩레이어 이름 정의
            mask_zero=True,                                     #mask_zero = True: max_len으로 시퀀스길이를 맞추기위해 0으로 패딩 -> 0으로 패딩된 값을 무시(패딩된 값이 학습이나 예측에 영향을 주지 않음)
            embeddings_regularizer=tf.keras.regularizers.L2(self.l2_reg), #아이템 임베딩 메트릭스 가중치에 L2정규화를 적용, 이 아이템 임베딩값이 학습하면서 변화함 그래서 초기값을 따로 지정해도 이 정규화는 유효함
        )
        # 평점(Rating) 전용 임베딩 층 (1~5점 + 패딩 0 = 6개)
        self.rating_embedding_layer = tf.keras.layers.Embedding(
            6, # input_dim: 0(padding), 1, 2, 3, 4, 5
            self.embedding_dim,
            name="rating_embeddings",
            mask_zero=True, # 평점 0은 무시
            embeddings_regularizer=tf.keras.regularizers.L2(self.l2_reg),
        )
        
        # 시간(Time) 전용 임베딩 층
        self.time_embedding_layer = tf.keras.layers.Embedding(
            self.max_interval, # input_dim: 0 ~ max_interval
            self.embedding_dim,
            name="time_embeddings",
            mask_zero=False, # 시간 0(첫 구매)은 의미가 있으므로 False
            embeddings_regularizer=tf.keras.regularizers.L2(self.l2_reg),
        )

        self.positional_embedding_layer = tf.keras.layers.Embedding(
            self.seq_max_len,                                  #seq_max_len: 포지셔널 임베딩레이어 입력차원
            self.embedding_dim,                                #embedding_dim: 포지셔널 임베딩레이어 출력차원
            name="positional_embeddings",
            mask_zero=False,
            embeddings_regularizer=tf.keras.regularizers.L2(self.l2_reg),
        )


        self.dropout_layer = tf.keras.layers.Dropout(self.dropout_rate)
        self.encoder = Encoder(
            self.num_blocks, #인코더레이어 개수
            self.seq_max_len,
            self.embedding_dim,
            self.attention_dim,
            self.attention_num_heads,
            self.conv_dims,
            self.dropout_rate,
        )

        self.decoder = Decoder(
            self.num_blocks, #인코더레이어 개수
            self.seq_max_len,
            self.embedding_dim,
            self.attention_dim,
            self.attention_num_heads,
            self.conv_dims,
            self.dropout_rate,
        )


        self.mask_layer = tf.keras.layers.Masking(mask_value=0) #인풋 시퀀스가 들어오면 mask value(0) 을 무시하고 처리함
        self.layer_normalization = LayerNormalization( #인코더레이어에서도 쓰고 여기서도 쓰고
            self.seq_max_len, self.embedding_dim, 1e-08
        )

    # 및 최신성(recency) 분석 로직 수정
    def embedding(self, input_seq, input_seq_time):
        """Compute the sequence and positional embeddings.
           아이템임베딩(시퀀스임베딩)과 포지셔널임베딩이 일어나는 부분
        Args:
            input_seq (tf.Tensor): Input sequence (B, S)
                                   S는 50 (학습) 또는 5 (평가)일 수 있음

        Returns:
            tf.Tensor, tf.Tensor:
            - Sequence embeddings.
            - Positional embeddings.
        """

        seq_embeddings = self.item_embedding_layer(input_seq) # (B, S, D)
        seq_embeddings = seq_embeddings * (self.embedding_dim ** 0.5)

        seq_time_embeddings = self.time_embedding_layer(input_seq_time) # (B, S, D)

        # 1. 현재 시퀀스 길이(S)를 동적으로 가져옴
        current_seq_len = tf.shape(input_seq)[1] # e.g., 50 (training) or 5 (eval)

        # 2. 올바른 위치 인덱스 범위를 생성
        #    (self.seq_max_len = 50 이라고 가정)
        #    S=50 (학습) -> start=0 -> range(0, 50) -> [0, ..., 49]
        #    S=5 (평가) -> start=45 -> range(45, 50) -> [45, 46, 47, 48, 49]
        start_index = self.seq_max_len - current_seq_len
        positional_seq_range = tf.range(start_index, self.seq_max_len) # (S,)

        # 3. (S,) -> (1, S)
        positional_seq = tf.expand_dims(positional_seq_range, 0)
        
        # 4. (1, S) -> (B, S)
        positional_seq = tf.tile(positional_seq, [tf.shape(input_seq)[0], 1]) 

        # 5. Keras 레이어를 호출
        #    - 첫 번째 호출(학습 시)에 레이어가 빌드
        #    - 평가 시 [45, ..., 49] 인덱스로 올바른 위치 임베딩을 가져옴
        positional_embeddings = self.positional_embedding_layer(positional_seq) # (B, S, D)

        positional_embeddings += seq_time_embeddings

        return seq_embeddings, positional_embeddings

    def embedding_rating(self, input_seq_rating):
        """레이팅만 임베딩
        Args:
            input_seq_rating (tf.Tensor): Input sequence

        Returns:
            tf.Tensor, tf.Tensor:
            - Sequence embeddings.
            - Positional embeddings.
        """

        seq_embeddings_rating = self.rating_embedding_layer(input_seq_rating) #인풋값을 임베딩벡터로 변환
        seq_embeddings_rating = seq_embeddings_rating * (self.embedding_dim ** 0.5) #값이 너무 커지는거 방지

        return seq_embeddings_rating

    def call(self, x, training):
        """Model forward pass.

        Args:
            x (tf.Tensor): Input tensor.
            training (tf.Tensor): Training tensor.

        Returns:
            tf.Tensor, tf.Tensor, tf.Tensor:
            - Logits of the positive examples.
            - Logits of the negative examples.
            - Mask for nonzero targets
        """

        input_seq = x["input_seq"]
        pos = x["positive"]
        neg = x["negative"]

        input_seq_rating = x["input_seq_rating"]
        pos_rating = x["positive_rating"]
        neg_rating = x["negative_rating"]

        input_seq_time = x["input_seq_time"]
        pos_time = x["positive_time"]
        neg_time = x["negative_time"]


        recent_item = self.recent_item * -1
        decoder_padding = self.seq_max_len - self.recent_item
        decoder_input_seq = tf.pad(input_seq[:, recent_item:], [[0, 0], [decoder_padding, 0]], mode='CONSTANT', constant_values=0)
        decoder_input_seq_rating = tf.pad(input_seq_rating[:, recent_item:], [[0, 0], [decoder_padding, 0]], mode='CONSTANT', constant_values=0)
        decoder_input_seq_time = tf.pad(input_seq_time[:, recent_item:], [[0, 0], [decoder_padding, 0]], mode='CONSTANT', constant_values=0)

        #mask
        mask = tf.expand_dims(tf.cast(tf.not_equal(input_seq, 0), tf.float32), -1) #tf.not_equal: input_seq에서 0이면 false, 0이아니면 true를 반환 -> tf.cast: false면 0, true면 1을 반환 -> expand_dims: 마지막차원에 새로운차원 추가
        mask_rating = tf.expand_dims(tf.cast(tf.not_equal(input_seq_rating, 0), tf.float32), -1)
        look_ahead_mask = 1 - tf.linalg.band_part(tf.ones((self.seq_max_len, self.seq_max_len)), -1, 0) #seq_len: 최근것을 얼마나 볼지



        # --- EMBEDDING --- (Encoder)
        seq_embeddings, positional_embeddings = self.embedding(input_seq, input_seq_time) #class 안에서 정의된 함수를 사용할때는 함수앞에 self.을 붙인다
        seq_embeddings_rating = self.embedding_rating(input_seq_rating) #포지셔널 임베딩은 위에꺼를 쓰는게 맞지않을까?

        # add positional embeddings
        seq_embeddings += positional_embeddings #시퀀스임베딩과 포지셔널임베딩을 더함
        seq_embeddings_rating += positional_embeddings

        # dropout
        seq_embeddings = self.dropout_layer(seq_embeddings) #시퀀스임베딩에 드롭아웃 적용
        seq_embeddings_rating = self.dropout_layer(seq_embeddings_rating)
        # masking
        seq_embeddings *= mask
        seq_embeddings_rating *= mask_rating


        # --- EMBEDDING ---(Decoder)
        seq_embeddings2, positional_embeddings2 = self.embedding(input_seq, input_seq_time) #class 안에서 정의된 함수를 사용할때는 함수앞에 self.을 붙인다
        seq_embeddings_rating2 = self.embedding_rating(input_seq_rating) #포지셔널 임베딩은 위에꺼를 쓰는게 맞지않을까?

        # add positional embeddings
        seq_embeddings2 += positional_embeddings2 #시퀀스임베딩과 포지셔널임베딩을 더함
        seq_embeddings_rating2 += positional_embeddings2

        # dropout
        seq_embeddings2 = self.dropout_layer(seq_embeddings2) #시퀀스임베딩에 드롭아웃 적용
        seq_embeddings_rating2 = self.dropout_layer(seq_embeddings_rating2)
        # masking
        seq_embeddings2 *= mask
        seq_embeddings_rating2 *= mask_rating




        # --- ATTENTION BLOCKS --- (Encoder)
        seq_attention = seq_embeddings
        seq_attention_rating = seq_embeddings_rating
        seq_attention = self.encoder(seq_attention, seq_attention_rating, training, mask) #셀프어텐션 -> FFN, 이 시점에서 인코더의 출력값
        seq_attention = self.layer_normalization(seq_attention)  # (b, s, d)

        # --- ATTENTION BLOCKS --- (Decoder)
        #seq_attention = self.decoder(seq_attention, seq_embeddings2, seq_embeddings_rating2, training, mask,look_ahead_mask) #디코더 사용시 해당 코드 주석해제



        # --- PREDICTION LAYER ---
        # user's sequence embedding
        pos = self.mask_layer(pos) #0을 무시하고 처리
        neg = self.mask_layer(neg)

        pos = tf.reshape(pos, [tf.shape(input_seq)[0] * self.seq_max_len]) #구매예정 리스트의 shape를 input의 배치사이즈*시퀀스최대길이(50)의 길이를 가지는 1차원 텐서로 변환
        neg = tf.reshape(neg, [tf.shape(input_seq)[0] * self.seq_max_len])
        pos_emb = self.item_embedding_layer(pos) #배치사이즈*시퀀스최대길이의 길이를 가지는 1차원 텐서로 reshape된 구매예정리스트를 임베딩레이어를 통과시킴
        neg_emb = self.item_embedding_layer(neg)
        seq_emb = tf.reshape( #인코더를 거친 output값을 (배치사이즈*시퀀스최대길이, 임베딩차원)으로 reshape
            seq_attention,
            [tf.shape(input_seq)[0] * self.seq_max_len, self.embedding_dim],
        )  # (b*s, d)

        pos_logits = tf.reduce_sum(pos_emb * seq_emb, -1) #구매예정 리스트 임베딩값 * 구매 리스트 임베딩값후 각 임베딩값을 더함 -> 곱할때 임베딩값이 일치할수록 로짓값이 최대
        neg_logits = tf.reduce_sum(neg_emb * seq_emb, -1)

        pos_logits = tf.expand_dims(pos_logits, axis=-1)  # (bs, 1)
        # pos_prob = tf.keras.layers.Dense(1, activation='sigmoid')(pos_logits)  # (bs, 1)

        neg_logits = tf.expand_dims(neg_logits, axis=-1)  # (bs, 1)
        # neg_prob = tf.keras.layers.Dense(1, activation='sigmoid')(neg_logits)  # (bs, 1)

        # output = tf.concat([pos_logits, neg_logits], axis=0)

        # masking for loss calculation
        istarget = tf.reshape(
            tf.cast(tf.not_equal(pos, 0), dtype=tf.float32), #pos의 각 요소가 0이 아니면 1, 0이면 0
            [tf.shape(input_seq)[0] * self.seq_max_len],
        )

        return pos_logits, neg_logits, istarget #sasrec 모델인스턴스의 리턴값: pos로짓값, neg로짓값, 구매예정리스트를 각요소가 0이면 0 아니면 1로 바꾼거(istarget)

    def predict(self, inputs):
        """Returns the logits for the test items.
           테스트 아이템에대한 로짓값을 리턴
        Args:
            inputs (tf.Tensor): Input tensor.

        Returns:
                 tf.Tensor: Output tensor.
        """
        training = False
        input_seq = inputs["input_seq"]
        candidate = inputs["candidate"]

        input_seq_rating = inputs["input_seq_rating"]
        input_seq_time = inputs["input_seq_time"]

        mask = tf.expand_dims(tf.cast(tf.not_equal(input_seq, 0), tf.float32), -1)
        mask_rating = tf.expand_dims(tf.cast(tf.not_equal(input_seq_rating, 0), tf.float32), -1)

        seq_embeddings, positional_embeddings = self.embedding(input_seq, input_seq_time)
        seq_embeddings_rating = self.embedding_rating(input_seq_rating)

        seq_embeddings += positional_embeddings
        seq_embeddings_rating += positional_embeddings # [수정] 평점 임베딩에도 위치 정보를 더하도록 수정 (학습 `call`과 일치)

        # seq_embeddings = self.dropout_layer(seq_embeddings)
        seq_embeddings *= mask
        seq_embeddings_rating *= mask_rating
        seq_attention = seq_embeddings
        seq_attention_rating = seq_embeddings_rating


        seq_attention = self.encoder(seq_attention, seq_attention_rating, training, mask) #1005
        seq_attention = self.layer_normalization(seq_attention)  # (b, s, d)
        
        
        batch_size = tf.shape(input_seq)[0
                                         
        sequence_length = tf.shape(input_seq)[1] # e.g., 5

        # (B, S, D) -> (B*S, D)
        seq_emb = tf.reshape(
            seq_attention,
            [batch_size * sequence_length, self.embedding_dim],
        )  # (b*s, d) (e.g., 5, 128)
        
        candidate_emb = self.item_embedding_layer(candidate)  # (b, num_neg+1, d)
        candidate_emb = tf.transpose(candidate_emb, perm=[0, 2, 1])  # (b, d, num_neg+1) (e.g., 1, 128, 101)

        # (b*s, d) @ (b, d, num_neg+1) -> (b, b*s, num_neg+1) (via broadcasting)
        # (5, 128) @ (1, 128, 101) -> (1, 5, 128) @ (1, 128, 101) -> (1, 5, 101)
        test_logits = tf.matmul(seq_emb, candidate_emb) 
        
        test_logits = tf.reshape(
            test_logits,
            [batch_size, sequence_length, 1 + self.num_neg_test],
        )  # (b, s, num_neg+1) (e.g., 1, 5, 101)
        
        test_logits = test_logits[:, -1, :]  # (b, num_neg+1) , 테스트니까 마지막 시퀀스의 값을 뽑음
        return test_logits

    def loss_function(self, pos_logits, neg_logits, istarget):
        """Losses are calculated separately for the positive and negative
        items based on the corresponding logits. A mask is included to
        take care of the zero items (added for padding).

        Args:
            pos_logits (tf.Tensor): Logits of the positive examples.
            neg_logits (tf.Tensor): Logits of the negative examples.
            istarget (tf.Tensor): Mask for nonzero targets.

        Returns:
            float: Loss.
        """

        pos_logits = pos_logits[:, 0] #그냥 pos_logit값임 끝에 0 신경안써ㅏ도되는듯 저거 차원확장으로 끝에 1 추가한거라서
        neg_logits = neg_logits[:, 0]

        # ignore padding items (0)
        # istarget = tf.reshape(
        #     tf.cast(tf.not_equal(self.pos, 0), dtype=tf.float32),
        #     [tf.shape(self.input_seq)[0] * self.seq_max_len],
        # )
        # for logits
        loss = tf.reduce_sum(
            -tf.math.log(tf.math.sigmoid(pos_logits) + 1e-24) * istarget
            - tf.math.log(1 - tf.math.sigmoid(neg_logits) + 1e-24) * istarget
        ) / tf.reduce_sum(istarget) #구매예정리스트(pos)와 구매하지않을것으로 예상되는 리스트(neg)에 대한 바이너리 크로스 엔트로피 손실 계산
        # for probabilities
        # loss = tf.reduce_sum(
        #         - tf.math.log(pos_logits + 1e-24) * istarget -
        #         tf.math.log(1 - neg_logits + 1e-24) * istarget
        # ) / tf.reduce_sum(istarget)
        reg_loss = tf.compat.v1.losses.get_regularization_loss() #규제값
        # reg_losses = tf.compat.v1.get_collection(tf.compat.v1.GraphKeys.REGULARIZATION_LOSSES)
        # loss += sum(reg_losses)
        loss += reg_loss

        return loss

    def create_combined_dataset(self, u, seq, pos, neg, seq_rating, pos_rating, neg_rating, seq_time, pos_time, neg_time):
        """
        function to create model inputs from sampled batch data.
        This function is used only during training.
        이건 학습할때만 사용됨
        """
        inputs = {}
        #maxlen길이만큼 0으로 패딩, 길면 앞쪽부터 잘라내기
        seq = tf.keras.preprocessing.sequence.pad_sequences(
            seq, padding="pre", truncating="pre", maxlen=self.seq_max_len
        )
        pos = tf.keras.preprocessing.sequence.pad_sequences(
            pos, padding="pre", truncating="pre", maxlen=self.seq_max_len
        )
        neg = tf.keras.preprocessing.sequence.pad_sequences(
            neg, padding="pre", truncating="pre", maxlen=self.seq_max_len
        )

        #maxlen길이만큼 0으로 패딩, 길면 앞쪽부터 잘라내기
        seq_rating = tf.keras.preprocessing.sequence.pad_sequences(
            seq_rating, padding="pre", truncating="pre", maxlen=self.seq_max_len
        )
        pos_rating = tf.keras.preprocessing.sequence.pad_sequences(
            pos_rating, padding="pre", truncating="pre", maxlen=self.seq_max_len
        )
        neg_rating = tf.keras.preprocessing.sequence.pad_sequences(
            neg_rating, padding="pre", truncating="pre", maxlen=self.seq_max_len
        )

        seq_time = tf.keras.preprocessing.sequence.pad_sequences(
            seq_time, padding="pre", truncating="pre", maxlen=self.seq_max_len
        )
        pos_time = tf.keras.preprocessing.sequence.pad_sequences(
            pos_time, padding="pre", truncating="pre", maxlen=self.seq_max_len
        )
        neg_time = tf.keras.preprocessing.sequence.pad_sequences(
            neg_time, padding="pre", truncating="pre", maxlen=self.seq_max_len
        )

        inputs["users"] = np.expand_dims(np.array(u), axis=-1)
        inputs["input_seq"] = seq
        inputs["positive"] = pos
        inputs["negative"] = neg

        inputs['input_seq_rating'] = seq_rating
        inputs['positive_rating'] = pos_rating
        inputs['negative_rating'] = neg_rating

        inputs['input_seq_time'] = seq_time
        inputs['positive_time'] = pos_time
        inputs['negative_time'] = neg_time

        target = np.concatenate(
            [
                np.repeat(1, seq.shape[0] * seq.shape[1]),
                np.repeat(0, seq.shape[0] * seq.shape[1]),
            ],
            axis=0,
        )
        target = np.expand_dims(target, axis=-1)
        return inputs, target #input: user, 패딩된 seq, pos,neg / target: 원래크기의 두배의 1과 0으로 된 np.array(2n,1)

    def train(self, dataset, sampler, **kwargs):
        """
        High level function for model training as well as
        evaluation on the validation and test dataset
        """
        num_epochs = kwargs.get("num_epochs", 10)
        batch_size = kwargs.get("batch_size", 128)
        lr = kwargs.get("learning_rate", 0.001)
        val_epoch = kwargs.get("val_epoch", 5)

        num_steps = int(len(dataset.user_train) / batch_size)

        optimizer = tf.keras.optimizers.Adam(
            learning_rate=lr, beta_1=0.9, beta_2=0.999, epsilon=1e-7
        )

        loss_function = self.loss_function #위에서 정의한 바이너리크로스엔트로피

        train_loss = tf.keras.metrics.Mean(name="train_loss") #나중에 각 배치에 대한 손실을 누적하고 평균으 계산하는데 사용됨

        #이러한 shape의 입력이 들어와야된다고 명시
        train_step_signature = [
            {
                "users": tf.TensorSpec(shape=(None, 1), dtype=tf.int64),
                "input_seq": tf.TensorSpec(
                    shape=(None, self.seq_max_len), dtype=tf.int64
                ),
                "positive": tf.TensorSpec(
                    shape=(None, self.seq_max_len), dtype=tf.int64
                ),
                "negative": tf.TensorSpec(
                    shape=(None, self.seq_max_len), dtype=tf.int64
                ),

                "input_seq_rating": tf.TensorSpec(
                    shape=(None, self.seq_max_len), dtype=tf.int64
                ),
                "positive_rating": tf.TensorSpec(
                    shape=(None, self.seq_max_len), dtype=tf.int64
                ),
                "negative_rating": tf.TensorSpec(
                    shape=(None, self.seq_max_len), dtype=tf.int64
                ),

                "input_seq_time": tf.TensorSpec(
                    shape=(None, self.seq_max_len), dtype=tf.int64
                ),
                "positive_time": tf.TensorSpec(
                    shape=(None, self.seq_max_len), dtype=tf.int64
                ),
                "negative_time": tf.TensorSpec(
                    shape=(None, self.seq_max_len), dtype=tf.int64
                ),
            },
            tf.TensorSpec(shape=(None, 1), dtype=tf.int64),
        ]

        @tf.function(input_signature=train_step_signature)
        def train_step(inp, tar):
            with tf.GradientTape() as tape: #그레디언트 계산
                pos_logits, neg_logits, loss_mask = self(inp, training=True) #sasrec 인스턴스사용, inp를 받아서 학습을 진행하며 로짓값과 loss_mask(이거 istarget인거같은데 -> 맞음) 반환
                loss = loss_function(pos_logits, neg_logits, loss_mask) #위에서 나온값을 넣어서 바이너리크로스엔트로피를 loss에 반환

            gradients = tape.gradient(loss, self.trainable_variables) #loss에 대한 모델의 가중치에 대한 그레디언트를 계산
            optimizer.apply_gradients(zip(gradients, self.trainable_variables)) #계산된 그레디언트 사용해서 사용된 옵티마이저(아담)을 통해 가중치 업데이트

            train_loss(loss) #학습중인 손실을 누적
            return loss #손실 반환

        T = 0.0
        t0 = Timer()
        t0.start()

        for epoch in range(1, num_epochs + 1):

            step_loss = []
            train_loss.reset_states()
            for step in tqdm(
                range(num_steps), total=num_steps, ncols=70, leave=False, unit="b"
            ):

                u, seq, pos, neg, seq_rating, pos_rating, neg_rating, seq_time, pos_time, neg_time = sampler.next_batch()

                inputs, target = self.create_combined_dataset(u, seq, pos, neg, seq_rating, pos_rating, neg_rating, seq_time, pos_time, neg_time)

                loss = train_step(inputs, target)
                step_loss.append(loss)

            if epoch % val_epoch == 0:
                t0.stop()
                t1 = t0.interval
                T += t1
                print("Evaluating...")
                
                # --- [수정됨] ---
                # (이전 수정과 동일)
                # 기본 평가(recency_window=None)를 호출합니다.
                # (이 값은 self.seq_max_len을 사용합니다)
                t_test = self.evaluate(dataset)
                t_valid = self.evaluate_valid(dataset)
                # ---------------
                
                print(
                    f"\nepoch: {epoch}, time: {T}, valid (NDCG@10: {t_valid[0]}, HR@10: {t_valid[1]})"
                )
                print(
                    f"epoch: {epoch}, time: {T},  test (NDCG@10: {t_test[0]}, HR@10: {t_test[1]})"
                )
                t0.start()

        t_test = self.evaluate(dataset)
        print(f"\nepoch: {epoch}, test (NDCG@10: {t_test[0]}, HR@10: {t_test[1]})")

        return t_test

    
    def evaluate(self, dataset, recency_window=None):
        """
        Evaluation on the test users (users with at least 3 items)
        
        Args:
            dataset: The dataset object.
            recency_window (int, optional): 
                If provided, analysis is performed using only the last N items 
                (N=recency_window) of the sequence. 
                If None, defaults to self.seq_max_len.
        """
        usernum = dataset.usernum
        itemnum = dataset.itemnum
        train = dataset.user_train  # removing deepcopy
        valid = dataset.user_valid
        test = dataset.user_test

        train_rating = dataset.user_train_rating
        valid_rating = dataset.user_valid_rating
        test_rating = dataset.user_test_rating

        train_time = dataset.user_train_time
        valid_time = dataset.user_valid_time
        test_time = dataset.user_test_time

        NDCG = 0.0
        HT = 0.0
        valid_user = 0.0

        # --- [수정됨] 리뷰어 #1, 7번 코멘트 반영 ---
        eval_max_len = self.seq_max_len # 50
        window_size = self.seq_max_len # 50 (default)
        
        if recency_window is not None:
            if recency_window > self.seq_max_len:
                print(f"Warning: recency_window ({recency_window}) > seq_max_len ({self.seq_max_len}). Clamping to {self.seq_max_len}.")
                window_size = self.seq_max_len
            else:
                window_size = recency_window
        # --------------------------------------

        if usernum > 10000:
            users = random.sample(range(1, usernum + 1), 10000)
        else:
            users = range(1, usernum + 1)

        for u in tqdm(users, ncols=70, leave=False, unit="b"):

            if len(train[u]) < 1 or len(test[u]) < 1:
                continue

            # --- [수정됨] 배열은 항상 (50,)으로 생성 ---
            seq = np.zeros([eval_max_len], dtype=np.int32)
            seq_rating = np.zeros([eval_max_len], dtype=np.int32)
            seq_time = np.zeros([eval_max_len], dtype=np.int32)
            idx = eval_max_len - 1 # 49
            # ----------------------------------------------

            
            # 1. valid[u][0] (test 직전 아이템)을 가장 마지막에(idx=49) 추가
            seq[idx] = valid[u][0]
            seq_rating[idx] = valid_rating[u][0]
            seq_time[idx] = valid_time[u][0]
            idx -= 1
            count = 1 # 1개 채움
            
            # 2. train[u]의 최근 아이템으로 (window_size - 1)개 만큼 마저 채움
            for i, k, t in zip(reversed(train[u]), reversed(train_rating[u]), reversed(train_time[u])):
                if count >= window_size: # e.g., 5개 찼으면 중지
                    break
                if idx < 0: # 배열의 시작(idx=0)까지 다 찼으면 중지
                    break
                    
                seq[idx] = i
                seq_rating[idx] = k
                seq_time[idx] = t
                idx -= 1
                count += 1
            
            # [수정 3] window_size 만큼만 잘라서 모델에 전달
            #    (e.g., seq[45:50])
            start_idx = eval_max_len - window_size
            
            input_seq_final = seq[start_idx:]
            input_seq_rating_final = seq_rating[start_idx:]
            input_seq_time_final = seq_time[start_idx:]

            rated = set(train[u])
            rated.add(0)
            item_idx = [test[u][0]]
            for _ in range(self.num_neg_test):
                t = np.random.randint(1, itemnum + 1)
                while t in rated:
                    t = np.random.randint(1, itemnum + 1)
                item_idx.append(t)

            inputs = {}
            inputs["user"] = np.expand_dims(np.array([u]), axis=-1)
            # (1, 50)이 아닌 (1, 5) 모양의 배열을 전달
            inputs["input_seq"] = np.array([input_seq_final]) 
            inputs["candidate"] = np.array([item_idx])

            inputs["input_seq_rating"] = np.array([input_seq_rating_final])
            inputs["input_seq_time"] = np.array([input_seq_time_final])

            # 'predict'는 이제 동적 길이를 지원합니다.
            predictions = -1.0 * self.predict(inputs)
            predictions = np.array(predictions)
            predictions = predictions[0]

            rank = predictions.argsort().argsort()[0]

            valid_user += 1

            if rank < 10:
                NDCG += 1 / np.log2(rank + 2)
                HT += 1

        return NDCG / valid_user, HT / valid_user


    def evaluate_valid(self, dataset, recency_window=None):
        """
        Evaluation on the validation users
        """
        usernum = dataset.usernum
        itemnum = dataset.itemnum
        train = dataset.user_train  # removing deepcopy
        valid = dataset.user_valid

        train_rating = dataset.user_train_rating
        valid_rating = dataset.user_valid_rating

        train_time = dataset.user_train_time
        valid_time = dataset.user_valid_time

        NDCG = 0.0
        valid_user = 0.0
        HT = 0.0
        

        eval_max_len = self.seq_max_len # 50
        window_size = self.seq_max_len # 50 (default)
        
        if recency_window is not None:
            if recency_window > self.seq_max_len:
                print(f"Warning: recency_window ({recency_window}) > seq_max_len ({self.seq_max_len}). Clamping to {self.seq_max_len}.")
                window_size = self.seq_max_len
            else:
                window_size = recency_window
        # --------------------------------------

        if usernum > 10000:
            users = random.sample(range(1, usernum + 1), 10000)
        else:
            users = range(1, usernum + 1)

        for u in tqdm(users, ncols=70, leave=False, unit="b"):
            if len(train[u]) < 1 or len(valid[u]) < 1:
                continue


            seq = np.zeros([eval_max_len], dtype=np.int32)
            seq_rating = np.zeros([eval_max_len], dtype=np.int32)
            seq_time = np.zeros([eval_max_len], dtype=np.int32)
            idx = eval_max_len - 1 # 49
            # ----------------------------------------------
            

            count = 0
            for i, k, t in zip(reversed(train[u]), reversed(train_rating[u]), reversed(train_time[u])):
                if count >= window_size: # e.g., 5개 찼으면 중지
                    break
                if idx < 0:
                    break
                    
                seq[idx] = i
                seq_rating[idx] = k
                seq_time[idx] = t
                idx -= 1
                count += 1
            

            start_idx = eval_max_len - window_size
            
            input_seq_final = seq[start_idx:]
            input_seq_rating_final = seq_rating[start_idx:]
            input_seq_time_final = seq_time[start_idx:]

            rated = set(train[u])
            rated.add(0)
            item_idx = [valid[u][0]] # 예측 대상 (정답)
            for _ in range(self.num_neg_test):
                t = np.random.randint(1, itemnum + 1)
                while t in rated:
                    t = np.random.randint(1, itemnum + 1)
                item_idx.append(t)

            inputs = {}
            inputs["user"] = np.expand_dims(np.array([u]), axis=-1)
            inputs["input_seq"] = np.array([input_seq_final]) # (1, 5)
            inputs["candidate"] = np.array([item_idx])

            inputs["input_seq_rating"] = np.array([input_seq_rating_final])
            inputs["input_seq_time"] = np.array([input_seq_time_final])

            # predictions = -model.predict(sess, [u], [seq], item_idx)
            predictions = -1.0 * self.predict(inputs)
            predictions = np.array(predictions)
            predictions = predictions[0]

            rank = predictions.argsort().argsort()[0]

            valid_user += 1

            if rank < 10:
                NDCG += 1 / np.log2(rank + 2)
                HT += 1

        return NDCG / valid_user, HT / valid_user

####sasrec util

In [50]:
# Copyright (c) Microsoft Corporation. All rights reserved.
# Licensed under the MIT License.
from collections import defaultdict


class SASRecDataSet:
    """
    A class for creating SASRec specific dataset used during
    train, validation and testing.

    Attributes:
        usernum: integer, total number of users
        itemnum: integer, total number of items
        User: dict, all the users (keys) with items as values
        Items: set of all the items
        user_train: dict, subset of User that are used for training
        user_valid: dict, subset of User that are used for validation
        user_test: dict, subset of User that are used for testing
        col_sep: column separator in the data file
        filename: data filename
    """

    def __init__(self, **kwargs):
        self.usernum = 0
        self.itemnum = 0
        self.User = defaultdict(list)
        self.Items = set()
        self.user_train = {}
        self.user_valid = {}
        self.user_test = {}
        self.col_sep = kwargs.get("col_sep", " ")
        self.filename = kwargs.get("filename", None)
        
        self.max_interval = 0
        
        if self.filename:
            with open(self.filename, "r") as fr:
                sample = fr.readline()
            ncols = sample.strip().split(self.col_sep)
            print(ncols)
            if ncols == 3:
                self.with_time = True
            else:
                self.with_time = True
            print(self.with_time)

    def split(self, item2vector=None, **kwargs):
        self.item2vector = item2vector
        self.filename = kwargs.get("filename", self.filename)
        if not self.filename:
            raise ValueError("Filename is required")

        if self.with_time:
            self.data_partition_with_time()
        else:
            self.data_partition()

    def data_partition(self):
        # assume user/item index starting from 1
        f = open(self.filename, "r")
        for line in f:
            u, i = line.rstrip().split(self.col_sep)
            u = int(u)
            i = int(i)
            self.usernum = max(u, self.usernum)
            self.itemnum = max(i, self.itemnum)
            i = self.item2vector[i]  # i 값을 해당 인덱스로 변경


            self.User[u].append(i)

        for user in self.User:
            nfeedback = len(self.User[user])
            if nfeedback < 3:
                self.user_train[user] = self.User[user]
                self.user_valid[user] = []
                self.user_test[user] = []
            else:
                self.user_train[user] = self.User[user][:-2]
                self.user_valid[user] = []
                self.user_valid[user].append(self.User[user][-2])
                self.user_test[user] = []
                self.user_test[user].append(self.User[user][-1])

    def data_partition_with_time(self):
        # assume user/item index starting from 1
        f = open(self.filename, "r")
        for line in f:
            u, i, t = line.rstrip().split(self.col_sep)
            u = int(u)
            i = int(i)
            t = float(t)
            self.usernum = max(u, self.usernum)
            self.itemnum = max(i, self.itemnum)
            self.User[u].append((i, t))
            self.Items.add(i)

        for user in self.User.keys():
            # sort by time
            items = sorted(self.User[user], key=lambda x: x[1])
            # keep only the items
            items = [x[0] for x in items]
            self.User[user] = items
            nfeedback = len(self.User[user])
            if nfeedback < 3:
                self.user_train[user] = self.User[user]
                self.user_valid[user] = []
                self.user_test[user] = []
            else:
                self.user_train[user] = self.User[user][:-2]
                self.user_valid[user] = []
                self.user_valid[user].append(self.User[user][-2])
                self.user_test[user] = []
                self.user_test[user].append(self.User[user][-1])

In [51]:
# Copyright (c) Microsoft Corporation. All rights reserved.
# Licensed under the MIT License.
from collections import defaultdict
from datetime import datetime

class SASRecDataSet_rating:
    """
    A class for creating SASRec specific dataset used during
    train, validation and testing.

    Attributes:
        usernum: integer, total number of users
        itemnum: integer, total number of items
        User: dict, all the users (keys) with items as values
        Items: set of all the items
        user_train: dict, subset of User that are used for training
        user_valid: dict, subset of User that are used for validation
        user_test: dict, subset of User that are used for testing
        col_sep: column separator in the data file
        filename: data filename
    """

    def __init__(self, **kwargs):
        self.usernum = 0
        self.itemnum = 0
        self.User = defaultdict(list)
        self.Items = set()
        self.user_train = {}
        self.user_valid = {}
        self.user_test = {}
        self.user_train_rating = {}
        self.user_valid_rating = {}
        self.user_test_rating = {}
        self.user_train_time = {}
        self.user_valid_time = {}
        self.user_test_time = {}
        self.col_sep = kwargs.get("col_sep", " ")
        self.filename = kwargs.get("filename", None)

        if self.filename:
            with open(self.filename, "r") as fr:
                sample = fr.readline()
            ncols = len(sample.strip().split(self.col_sep))
            print(ncols)
            if ncols == 4:
                self.with_time = True
            else:
                self.with_time = False

    def split(self, item2vector=None, **kwargs):
        self.item2vector = item2vector
        self.filename = kwargs.get("filename", self.filename)
        if not self.filename:
            raise ValueError("Filename is required")

        if self.with_time:
            self.data_partition_with_time()
        else:
            self.data_partition()

    def data_partition(self):
        # assume user/item index starting from 1
        f = open(self.filename, "r")
        for line in f:
            u, i, r = line.rstrip().split(self.col_sep)
            u = int(u)
            i = int(i)
            r = float(r)
            self.usernum = max(u, self.usernum)
            self.itemnum = max(i, self.itemnum)

            self.User[u].append((i, r))

        for user in self.User:
            items_ratings = self.User[user]
            nfeedback = len(items_ratings)

            items = [ir[0] for ir in items_ratings]
            ratings = [ir[1] for ir in items_ratings]

            if nfeedback < 3:
                self.user_train[user] = items
                self.user_valid[user] = []
                self.user_test[user] = []

                self.user_train_rating[user] = ratings
                self.user_valid_rating[user] = []
                self.user_test_rating[user] = []
            else:
                self.user_train[user] = items[:-2]
                self.user_valid[user] = [items[-2]]
                self.user_test[user] = [items[-1]]

                self.user_train_rating[user] = ratings[:-2]
                self.user_valid_rating[user] = [ratings[-2]]
                self.user_test_rating[user] = [ratings[-1]]

    def data_partition_with_time(self):
        # Date format in the file
        date_format = "%Y-%m-%d"
        base_date = 0  # Use 1 for times less than a day

        # assume user/item index starting from 1
        f = open(self.filename, "r")
        user_time_intervals = {}  # 유저별 구매 간격 저장을 위한 딕셔너리
        for line in f:
            u, i, r, t = line.rstrip().split(self.col_sep)
            u = int(u)
            i = int(i)
            r = float(r)
            t = datetime.strptime(t, "%Y-%m-%d").date()

            if u not in user_time_intervals:
                user_time_intervals[u] = [t]
            else:
                user_time_intervals[u].append(t)

            self.usernum = max(u, self.usernum)
            self.itemnum = max(i, self.itemnum)
            self.User[u].append((i, r, t))

        # 구매 간격을 계산
        all_intervals = []  # 모든 유저의 구매 간격 저장을 위한 리스트
        for user, dates in user_time_intervals.items():
            dates.sort()  # 시간을 오름차순으로 정렬
            intervals = [0]  # 첫 구매는 0으로 시작
            for j in range(1, len(dates)):
                interval = (dates[j] - dates[j-1]).days
                if interval < 0:
                    print(f"Error: Negative interval for user {user} between {dates[j-1]} and {dates[j]}")
                intervals.append(max(0, interval))  # 음수 간격 방지
                all_intervals.append(max(0, interval))
            user_time_intervals[user] = intervals

        # Min-Max Scaling
        min_interval = min(all_intervals)
        self.max_interval = max(all_intervals) # 'max_interval'을 'self.max_interval'로 변경

        for user, intervals in user_time_intervals.items():
            #normalized_intervals = [(interval - min_interval) / (max_interval - min_interval) for interval in intervals]
            #user_time_intervals[user] = normalized_intervals
            user_time_intervals[user] = intervals
        for user in self.User:
            items_ratings_times = self.User[user]
            items_ratings_times.sort(key=lambda x: x[2])  # 시간으로 정렬

            items = [irt[0] for irt in items_ratings_times]
            ratings = [irt[1] for irt in items_ratings_times]
            times = user_time_intervals[user]

            nfeedback = len(items_ratings_times)

            if nfeedback < 3:
                self.user_train[user] = items
                self.user_train_rating[user] = ratings
                self.user_train_time[user] = times

                self.user_valid[user] = []
                self.user_valid_rating[user] = []
                self.user_valid_time[user] = []

                self.user_test[user] = []
                self.user_test_rating[user] = []
                self.user_test_time[user] = []

            else:
                self.user_train[user] = items[:-2]
                self.user_train_rating[user] = ratings[:-2]
                self.user_train_time[user] = times[:-2]

                self.user_valid[user] = [items[-2]]
                self.user_valid_rating[user] = [ratings[-2]]
                self.user_valid_time[user] = [times[-2]]

                self.user_test[user] = [items[-1]]
                self.user_test_rating[user] = [ratings[-1]]
                self.user_test_time[user] = [times[-1]]

####sampler(Custom)

In [52]:
#sampler = WarpSampler(data.user_train, data.usernum, data.itemnum, batch_size=batch_size, maxlen=maxlen, n_workers=3)
import numpy as np
from multiprocessing import Process, Queue


def random_neq(left, right, s):
    t = np.random.randint(left, right)
    while t in s:
        t = np.random.randint(left, right)
    return t


def sample_function(
    user_train, user_train_rating, user_train_time, usernum, itemnum, batch_size, maxlen, result_queue, seed
):
    """Batch sampler that creates a sequence of negative items based on the
    original sequence of items (positive) that the user has interacted with.

    Args:
        user_train (dict): dictionary of training exampled for each user
        usernum (int): number of users
        itemnum (int): number of items
        batch_size (int): batch size
        maxlen (int): maximum input sequence length
        result_queue (multiprocessing.Queue): queue for storing sample results
        seed (int): seed for random generator
    """

    def sample():

        user = np.random.randint(1, usernum + 1) #유저중 한명 선택
        while len(user_train[user]) <= 1:
            user = np.random.randint(1, usernum + 1)

        seq = np.zeros([maxlen], dtype=np.int32) #maxlen(50)만큼의 0으로 된 np 생성
        pos = np.zeros([maxlen], dtype=np.int32)
        neg = np.zeros([maxlen], dtype=np.int32)
        nxt = user_train[user][-1] #선택된 유저의 train 구매리스트에서 가장 마지막거를 nxt에 넣음

        seq_rating = np.zeros([maxlen], dtype=np.int32) #maxlen(50)만큼의 0으로 된 np 생성
        pos_rating = np.zeros([maxlen], dtype=np.int32)
        neg_rating = np.zeros([maxlen], dtype=np.int32)
        nxt_rating = user_train_rating[user][-1]

        seq_time = np.zeros([maxlen], dtype=np.int32) #maxlen(50)만큼의 0으로 된 np 생성
        pos_time = np.zeros([maxlen], dtype=np.int32)
        neg_time = np.zeros([maxlen], dtype=np.int32)
        nxt_time = user_train_time[user][-1]

        idx = maxlen - 1 #49




        ts = set(list(user_train[user])) #선택된 유저의 train 구매리스트를 list화하고 집합으로 변경
        ts_rating = list(user_train_rating[user])
        ts_time = list(user_train_time[user])
        for i, k, t in zip(reversed(user_train[user][:-1]), reversed(user_train_rating[user][:-1]), reversed(user_train_time[user][:-1])): #선택된 유저의 train 구매리스트에서 마지막거를 제외한 나머지를 역순으로 i에 넣어서 for문 (1,2,3,4,5,6,7,8,9,10) 이면 9,8,7,6,5,4,3,2,1
            seq[idx] = i #
            pos[idx] = nxt #nxt: 선택된 유저의 train 구매리스트에서 가장 마지막거

            seq_rating[idx] = k
            pos_rating[idx] = nxt_rating

            seq_time[idx] = t
            pos_time[idx] = nxt_time
            if nxt != 0: #nxt가 0이 아니면
                neg[idx] = random_neq(1, itemnum + 1, ts) #선택된 유저의 train 구매리스트에 포함되지않는(구매를 안한)item을 idx번째에 넣음
            if nxt_rating != 0: #nxt가 0이 아니면
                neg_rating[idx] = float(0.0000001)  #
            if nxt_time >= 0: # 0일 수도 있으므로 >= 로 변경
                neg_time[idx] = nxt_time # <--- nxt_time (직전의 실제 시간)

            nxt = i #10 -> 9
            nxt_rating = k
            nxt_time = t
            idx -= 1 #49 -> 48
            if idx == -1: #-1되면 멈추는거때문에 seq pos neg의 0번째가 할당되면 끝남
                break

        return (user, seq, pos, neg, seq_rating, pos_rating, neg_rating, seq_time, pos_time, neg_time)

    np.random.seed(seed)
    while True:
        one_batch = []
        for i in range(batch_size):
            one_batch.append(sample()) #one_batch에 batch_size만큼 user,seq,pos,neg가 생김, neg만 랜덤하게 바뀜 나머진 안변하고

        result_queue.put(zip(*one_batch))


class WarpSampler(object):
    """Sampler object that creates an iterator for feeding batch data while training.

    Attributes:
        User: dict, all the users (keys) with items as values
        usernum: integer, total number of users
        itemnum: integer, total number of items
        batch_size (int): batch size
        maxlen (int): maximum input sequence length
        n_workers (int): number of workers for parallel execution
    """

    def __init__(self, User, User_rating, User_time, usernum, itemnum, batch_size=64, maxlen=10, n_workers=1):
        self.result_queue = Queue(maxsize=n_workers * 10)
        self.processors = []
        for i in range(n_workers):
            self.processors.append(
                Process(
                    target=sample_function,
                    args=(
                        User,
                        User_rating,
                        User_time,
                        usernum,
                        itemnum,
                        batch_size,
                        maxlen,
                        self.result_queue,
                        np.random.randint(2e9),
                    ),
                )
            )
            self.processors[-1].daemon = True
            self.processors[-1].start()

    def next_batch(self):
        return self.result_queue.get()

    def close(self):
        for p in self.processors:
            p.terminate()
            p.join()

### Input Parameters

In [58]:
num_epochs = 100
batch_size = 256
RANDOM_SEED = 101  # Set None for non-deterministic result

# data_dir = os.path.join("tests", "recsys_data", "RecSys", "SASRec-tf2", "data")
data_dir = os.path.join("..", "..", "tests", "resources", "deeprec", "sasrec")

# Amazon Electronics Data
#dataset = "reviews_Electronics_5"

lr = 0.001             # learning rate
maxlen = 50            # maximum sequence length for each user
num_blocks = 2         # 2
hidden_units = 128    # 128
num_heads = 1         # 1
dropout_rate = 0.5     # dropout rate
l2_emb = 0       # 0.0 -> 1e-5 (과적합 방지)
num_neg_test = 100     # number of negative examples per positive example
recent_item = 5
model_name = 'sasrec'  # 'sasrec' or 'ssept'

#rating Input

In [59]:
#inp_file = os.path.join(data_dir, dataset + ".txt")
inp_file = 'data/Beautyduple.txt'
print(inp_file)

# initiate a dataset class
data = SASRecDataSet_rating(filename=inp_file, col_sep=" ")

# create train, validation and test splits
#data.split(item2vector=item_vectors)
data.split()

# some statistics
num_steps = int(len(data.user_train) / batch_size)
cc = 0.0
for u in data.user_train:
    cc += len(data.user_train[u])
print('%g Users and %g items' % (data.usernum, data.itemnum))
print('average sequence length: %.2f' % (cc / len(data.user_train)))

data/Beautyduple.txt
4
6107 Users and 5514 items
average sequence length: 5.16


### Model Creation

Model parameters are

    - number of items
    - maximum sequence length of the user interaction history
    - number of Transformer blocks
    - embedding dimension for item embedding
    - dimension of the attention
    - number of attention heads
    - dropout rate
    - dimension of the convolution layers, list
    - L_2-regularization coefficient

In [60]:
if model_name == 'sasrec':
    model = SASREC(item_num=data.itemnum,
                   seq_max_len=maxlen,
                   max_interval=data.max_interval + 1, # +1 (for padding 0)
                   num_blocks=num_blocks,
                   embedding_dim=hidden_units,
                   attention_dim=hidden_units,
                   attention_num_heads=num_heads,
                   dropout_rate=dropout_rate,
                   conv_dims = [hidden_units, hidden_units],
                   l2_reg=l2_emb,
                   recent_item = recent_item,
                   num_neg_test=num_neg_test
    )
elif model_name == "ssept":
    model = SSEPT(item_num=data.itemnum,
                  user_num=data.usernum,
                  seq_max_len=maxlen,
                  num_blocks=num_blocks,
                  # embedding_dim=hidden_units,  # optional
                  user_embedding_dim=10,
                  item_embedding_dim=hidden_units,
                  attention_dim=hidden_units,
                  attention_num_heads=num_heads,
                  dropout_rate=dropout_rate,
                  conv_dims = [110, 110],
                  l2_reg=l2_emb,
                  num_neg_test=num_neg_test
    )
else:
    print(f"Model-{model_name} not found")

### Sampler

    - the sampler creates negative samples from the training data for each batch
    - this is done by looking at the original user interaction history and creating items that are not present at all
    - the sampler generates a sequence of negative items of the same length as the original history

In [61]:
#sampler = WarpSampler(data.user_train, data.usernum, data.itemnum, batch_size=batch_size, maxlen=maxlen, n_workers=3) #rating없이 하는거
sampler = WarpSampler(data.user_train, data.user_train_rating, data.user_train_time, data.usernum, data.itemnum, batch_size=batch_size, maxlen=maxlen, n_workers=8)

### Model Training

    - the loss function is defined over all the negative and positive logits
    - a mask has to be applied to indicate the non-zero items present in the output
    - we also add the regularization loss here
    
    - having a train-step signature function can speed up the training process

In [62]:
with Timer() as train_time:
    t_test = model.train(data, sampler, num_epochs=num_epochs, batch_size=batch_size, lr=lr, val_epoch=100)

print('Time cost for training is {0:.2f} mins'.format(train_time.interval/60.0))

Evaluating...



epoch: 100, time: 142.19386429400765, valid (NDCG@10: 0.5330407498743263, HR@10: 0.6810344827586207)
epoch: 100, time: 142.19386429400765,  test (NDCG@10: 0.4870251461574721, HR@10: 0.6346153846153846)



epoch: 100, test (NDCG@10: 0.4876698682437138, HR@10: 0.6354442970822282)
Time cost for training is 8.39 mins


In [15]:
res_syn = {"ndcg@10": t_test[0], "Hit@10": t_test[1]}
print(res_syn)

{'ndcg@10': 0.48676284004480497, 'Hit@10': 0.6269893899204244}


In [63]:

recency_windows = [5, 10, 20, 30, 40, 50]

results = []

print("시퀀스 길이 조절후 성능평가")

for window_size in recency_windows:
    print(f"\nEvaluating with recency_window = {window_size}...")
    
    # 1. Validation Set 평가
    t_valid = model.evaluate_valid(data, recency_window=window_size)
    
    # 2. Test Set 평가
    t_test = model.evaluate(data, recency_window=window_size)
    
    print(f"Window: {window_size}, Valid (NDCG@10: {t_valid[0]:.4f}, HR@10: {t_valid[1]:.4f})")
    print(f"Window: {window_size}, Test (NDCG@10: {t_test[0]:.4f}, HR@10: {t_test[1]:.4f})")
    
    results.append({
        "window": window_size,
        "valid_ndcg": t_valid[0],
        "valid_hr": t_valid[1],
        "test_ndcg": t_test[0],
        "test_hr": t_test[1]
    })

print("\n--- 평가 완료 ---")

# 결과 출력
try:
    import pandas as pd
    df_results = pd.DataFrame(results).set_index('window')
    print(df_results)
except ImportError:
    print(results)


--- [리뷰어 #1, 7번] 최신성 창(Recency Window) 분석 시작 ---
모델 재훈련 없이, 평가 시점의 입력 시퀀스 길이만 조절하여 성능을 분석합니다.

Evaluating with recency_window = 5...


Window: 5, Valid (NDCG@10: 0.5308, HR@10: 0.6767)
Window: 5, Test (NDCG@10: 0.4846, HR@10: 0.6328)

Evaluating with recency_window = 10...


Window: 10, Valid (NDCG@10: 0.5300, HR@10: 0.6795)
Window: 10, Test (NDCG@10: 0.4878, HR@10: 0.6353)

Evaluating with recency_window = 20...


Window: 20, Valid (NDCG@10: 0.5309, HR@10: 0.6774)
Window: 20, Test (NDCG@10: 0.4863, HR@10: 0.6330)

Evaluating with recency_window = 30...


Window: 30, Valid (NDCG@10: 0.5303, HR@10: 0.6764)
Window: 30, Test (NDCG@10: 0.4888, HR@10: 0.6358)

Evaluating with recency_window = 40...


Window: 40, Valid (NDCG@10: 0.5327, HR@10: 0.6792)
Window: 40, Test (NDCG@10: 0.4855, HR@10: 0.6351)

Evaluating with recency_window = 50...


Window: 50, Valid (NDCG@10: 0.5340, HR@10: 0.6781)
Window: 50, Test (NDCG@10: 0.4886, HR@10: 0.6353)

--- 분석 완료 ---
        valid_ndcg  valid_hr  test_ndcg   test_hr
window                                           
5         0.530823  0.676724   0.484570  0.632792
10        0.529961  0.679542   0.487751  0.635279
20        0.530899  0.677387   0.486347  0.632958
30        0.530278  0.676393   0.488754  0.635776
40        0.532708  0.679211   0.485472  0.635113
50        0.533950  0.678050   0.488567  0.635279


In [ ]:
# [새로운 셀 - 리뷰어 #1, 7번 코멘트(실용적 가이던스) 분석용]
#
# 이 셀은 Cell 11에서 모델 학습이 완료된 *이후*에 실행해야 합니다.
# (모델을 재학습할 필요가 없습니다)

print("\n--- [리뷰어 #1, 7번] 최신성 창(Recency Window) 분석 시작 ---")

# 분석할 최신성 창 크기 (시퀀스 절단 길이)
# (예: 마지막 5, 10, 20, 30, 40, 50개 아이템)
# 50은 기본 maxlen과 동일합니다.
recency_windows = [5, 10, 20, 30, 40, 50]

results = []

print("모델 재훈련 없이, 평가 시점의 입력 시퀀스 길이만 조절하여 성능을 분석합니다.")

for window_size in recency_windows:
    print(f"\nEvaluating with recency_window = {window_size}...")
    
    # 1. Validation Set 평가
    t_valid = model.evaluate_valid(data, recency_window=window_size)
    
    # 2. Test Set 평가
    t_test = model.evaluate(data, recency_window=window_size)
    
    print(f"Window: {window_size}, Valid (NDCG@10: {t_valid[0]:.4f}, HR@10: {t_valid[1]:.4f})")
    print(f"Window: {window_size}, Test (NDCG@10: {t_test[0]:.4f}, HR@10: {t_test[1]:.4f})")
    
    results.append({
        "window": window_size,
        "valid_ndcg": t_valid[0],
        "valid_hr": t_valid[1],
        "test_ndcg": t_test[0],
        "test_hr": t_test[1]
    })

print("\n--- 분석 완료 ---")

# 결과 출력 (e.g., Pandas DataFrame으로)
try:
    import pandas as pd
    df_results = pd.DataFrame(results).set_index('window')
    print(df_results)
except ImportError:
    print(results)

## Reference
\[1\] Wang-Cheng Kang, Julian McAuley: Self-Attentive Sequential Recommendation, arXiv preprint arXiv:1808.09781 (2018) <br>

\[2\] Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N Gomez, Łukasz Kaiser, and Illia Polosukhin. 2017. Attention is all you need. In Advances in Neural Information Processing Systems. 5998–6008 <br>

\[3\] Jiaxi Tang and Ke Wang. 2018. Personalized top-n sequential recommendation via convolutional sequence embedding. In Proceedings of the Eleventh ACM International Conference on Web Search and Data Mining. ACM, 565–573.

\[4\] Balázs Hidasi, Alexandros Karatzoglou, Linas Baltrunas, and Domonkos Tikk. 2015. Session-based recommendations with recurrent neural networks. arXiv preprint arXiv:1511.06939 (2015)

\[5\] Zeping Yu, Jianxun Lian, Ahmad Mahmoody, Gongshen Liu, Xing Xie. Adaptive User Modeling with Long and Short-Term Preferences for Personailzed Recommendation. In Proceedings of the 28th International Joint Conferences on Artificial Intelligence, IJCAI’19, Pages 4213-4219. AAAI Press, 2019.

\[6\] Liwei Wu, Shuqing Li, Cho-Jui Hsieh, James Sharpnack. SSE-PT: Sequential Recommendation Via Personalized Transformer. In Fourteenth ACM Conference on Recommender Systems, RecSys'20:, Pages 328–337, 2020.